In [29]:
import numpy as np
from gensim.models import Word2Vec
from nltk.tokenize import sent_tokenize, word_tokenize

In [30]:
with open("../data/my_sweet_orange_tree.txt", "r", encoding="utf-8") as file:
    raw_text = file.read()

In [31]:
# tokenization

In [32]:
sentences = sent_tokenize(raw_text[:10000].lower())
tokenized_sentences = [word_tokenize(sentence) for sentence in sentences]

In [33]:
print(f"sentences size: {len(sentences)}")
print(f"token size: {len(tokenized_sentences)}")

sentences size: 210
token size: 210


In [34]:
embedding_model = Word2Vec(sentences=tokenized_sentences, vector_size=100, window=5, min_count=1, sg=1, workers=4, epochs=10)

In [35]:
len(embedding_model.wv.index_to_key)

577

In [36]:
def create_sequence(sentences):
    pairs = []

    for sentence in sentences:
        for i in range(1, len(sentence)):
            context = sentence[:i]
            target= sentence[i]

            pairs.append((context, target))

    return pairs

In [37]:
def context_vector(context, embedding_model):
    vectors = [
        embedding_model.wv[word]
        for word in context
        if word in embedding_model.wv
    ]

    if len(vectors) == 0:
        return np.zeros(100)
    
    return np.mean(vectors, axis=0)

In [9]:
def context_to_vector(context, embedding_model, window=3):
    context = context[-window:]
    vectors = [
        embedding_model.wv[word]
        for word in context
    ]

    # Pad missing context at the beginning
    while len(vectors) < window:
        vectors.insert(
            0,
            np.zeros(embedding_model.vector_size)
        )

    # Preserve word order
    return np.concatenate(vectors)

In [38]:
pairs = create_sequence(tokenized_sentences)

In [39]:
word_to_idx = {
    word: idx
    for idx, word in enumerate(embedding_model.wv.index_to_key)
}

In [40]:
idx_to_word = {
    idx: word
    for word, idx in word_to_idx.items()
}

In [41]:
X = np.array([
    context_to_vector(context, embedding_model)
    for context, target in pairs
])
y = np.array([
    target
    for _, target in pairs
])

y = np.array([
    word_to_idx[word]
    for word in y
])

In [42]:
print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")

X shape: (2241, 300)
y shape: (2241,)


In [43]:
def onehot(Y):
    return np.eye(len(word_to_idx))[Y]

In [44]:
# relu
def relu(Z):
    return np.maximum(0, Z)
# softmax
def softmax(Z):
    exp_z = np.exp(Z - np.max(Z, axis=1, keepdims=True))
    return exp_z / np.sum(exp_z, axis=1, keepdims=True)

In [45]:
def initialize_parameters(
    input_size, hiddenLayer1_size, hiddenLayer2_size, output_size
):
    np.random.seed(42)

    parameters = {
        "w1": np.random.randn(input_size, hiddenLayer1_size) * np.sqrt(2 / input_size),
        "b1": np.zeros((1, hiddenLayer1_size)),
        "w2": np.random.randn(hiddenLayer1_size, hiddenLayer2_size) * np.sqrt(2 / hiddenLayer1_size),
        "b2": np.zeros((1, hiddenLayer2_size)),
        "w3": np.random.randn(hiddenLayer2_size, output_size) * np.sqrt(2 / hiddenLayer2_size),
        "b3": np.zeros((1, output_size)),
    }

    return parameters

In [46]:
np.random.randn(2,3)

array([[-0.13508281, -0.15645202, -0.77950097],
       [ 0.45182494,  1.19605675, -0.47474716]])

In [47]:
def forward_propagation(X, parameters):
    w1, b1, w2, b2, w3, b3 = (
        parameters["w1"],
        parameters["b1"],
        parameters["w2"],
        parameters["b2"],
        parameters["w3"],
        parameters["b3"],
    )

    # first layer
    Z1 = np.dot(X, w1) + b1
    A1 = relu(Z1)

    # second layer
    Z2 = np.dot(A1, w2) + b2
    A2 = relu(Z2)

    # output layer
    Z3 = np.dot(A2, w3) + b3
    A3 = softmax(Z3)

    cache = {
        "Z1": Z1,
        "A1": A1,
        "Z2": Z2,
        "A2": A2,
        "Z3": Z3,
        "A3": A3,
    }

    return A3, cache


In [48]:
def compute_cost(Y, A3):
    m = Y.shape[0]

    correct_prob = np.clip(A3[np.arange(m), Y], 1e-12, 1.0)

    L = -1 / m * np.sum(np.log(correct_prob))

    return L

In [49]:
def backpropagation(w1, b1, w2, b2, w3, b3, cache, X, Y, alpha):
    m = Y.shape[0]
    Z1, A1, Z2, A2, Z3, A3 = cache["Z1"], cache["A1"], cache["Z2"], cache["A2"], cache["Z3"], cache["A3"]

    # output layer
    dZ3 = A3 - onehot(Y)
    dW3 = np.dot(A2.T, dZ3) / m
    db3 = np.sum(dZ3, axis=0, keepdims=True) / m

    # second layer
    dA2 = np.dot(dZ3, w3.T)
    dZ2 = dA2 * (Z2 > 0) # relu'
    db2 = np.sum(dZ2, axis=0, keepdims=True) / m
    dW2 = np.dot(A1.T, dZ2) / m

    # first layer
    dA1 = np.dot(dZ2, w2.T)
    dZ1 = dA1 * (Z1 > 0)
    db1 = np.sum(dZ1, axis=0, keepdims=True) / m
    dW1 = np.dot(X.T, dZ1) / m
    
    # print(dZ3.shape) # (4316, 1080)
    # print(dW3.shape) # (6, 1080)
    # print(db3.shape) # (1080,)
    # print()
    
    # print(dA2.shape) # (4316, 6)
    # print(dZ2.shape) # (4316, 6)
    # print(db2.shape) # (1080,)
    # print(dW2.shape) # (12, 6)
    # print()

    # print(dA1.shape) # (4316, 12)
    # print(dZ1.shape) # (4316, 12)
    # print(db1.shape) # (12,)
    # print(dW1.shape) # (100, 12)

    return dW1, db1, dW2, db2, dW3, db3

In [50]:
input_size = embedding_model.wv.vector_size * 3
hidden1_size = 128
hidden2_size = 64
output_size = len(embedding_model.wv.index_to_key)

params = initialize_parameters(input_size, hidden1_size, hidden2_size, output_size)

In [51]:
print(f"y shape: {y.shape}")
print(f"X shape: {X.shape}")

y shape: (2241,)
X shape: (2241, 300)


In [24]:
A3, cache = forward_propagation(X, params)

In [25]:
for key, val in cache.items():
    print(f"{key} shape: {val.shape}")

print()
for key, val in params.items():
    print(f"{key} shape: {val.shape}")    

Z1 shape: (1111, 128)
A1 shape: (1111, 128)
Z2 shape: (1111, 64)
A2 shape: (1111, 64)
Z3 shape: (1111, 372)
A3 shape: (1111, 372)

w1 shape: (300, 128)
b1 shape: (1, 128)
w2 shape: (128, 64)
b2 shape: (1, 64)
w3 shape: (64, 372)
b3 shape: (1, 372)


In [27]:
alpha = 0.01
costs = []
for i in range(500):
    A3, cache = forward_propagation(X, params)
    cost = compute_cost(y, A3)
    costs.append(cost)
    dw1, db1, dw2, db2, dw3, db3 = backpropagation(params["w1"], params["b1"], params["w2"], params["b2"], params["w3"], params["b3"], cache, X, y, 0.05)

    # update params
    params["w1"] -= alpha * dw1
    params["b1"] -= alpha * db1
    
    params["w2"] -= alpha * dw2
    params["b2"] -= alpha * db2
    
    params["w3"] -= alpha * dw3
    params["b3"] -= alpha * db3

    if i % 50 == 0:
        print(f"epoch {i}, lost: {cost}")

epoch 0, lost: 5.7066016392912005
epoch 50, lost: 5.6731103795202715
epoch 100, lost: 5.635335768686027
epoch 150, lost: 5.593218601287809
epoch 200, lost: 5.5473680543300805
epoch 250, lost: 5.499716341520048



KeyboardInterrupt



KeyboardInterrupt: 

In [53]:
import matplotlib.pyplot as plt

In [ ]:
plt.plot(costs)
plt.xlabel("cost")
plt.ylabel("training cost")
plt.show()

In [213]:
# Check the training results
assert np.all(np.isfinite(costs))
assert np.all(np.isfinite(params["w1"]))
assert np.all(np.isfinite(params["w2"]))
assert np.all(np.isfinite(params["w3"]))

print("Initial cost:", costs[0])
print("Final cost:", costs[-1])
print("Cost decreased:", costs[-1] < costs[0])

Initial cost: 5.920674668471389
Final cost: 5.881108109072329
Cost decreased: True


In [52]:
def predict_next_word(context):
    context = context.lower().split()

    if not context:
        raise ValueError("Provide at least one context word.")

    # Check that all context words exist in the vocabulary
    unknown = [word for word in context if word not in word_to_idx]
    if unknown:
        raise ValueError(f"Unknown vocabulary words: {unknown}")

    x = context_to_vector(context, embedding_model).reshape(1, -1)
    probabilities, _ = forward_propagation(x, params)

    predicted_idx = int(np.argmax(probabilities[0]))
    return idx_to_word[predicted_idx]


print(predict_next_word("sometimes"))
# print(predict_next_word("he was"))
# print(predict_next_word("it was a"))

she


In [67]:
predictions, _ = forward_propagation(X[:10], params)

print("Output probabilities for first example:")
top_indices = np.argsort(predictions[0])[-10:][::-1]

for idx in top_indices:
    print(idx_to_word[int(idx)], predictions[0][idx])

print("\nMean output probability for comma:",
      predictions[:, word_to_idx[',']].mean())

print("Mean maximum probability:",
      predictions.max(axis=1).mean())

Output probabilities for first example:
chapter 0.002751435005592695
brother 0.002739251326364823
using 0.002736434728847733
must 0.0027360125970452074
only 0.002730249261494739
different 0.002728173532943042
matter 0.0027281358631408973
already 0.0027257075892685436
a 0.0027246629115152527
teaching 0.0027241786530147255

Mean output probability for comma: 0.002956235002443576
Mean maximum probability: 0.003451078968290393


In [68]:
from collections import Counter

target_counts = Counter(y)

print("Most common target words:")
for idx, count in target_counts.most_common(15):
    print(f"{idx_to_word[idx]!r}: {count}")

print("\nComma target frequency:",
      target_counts[word_to_idx[',']] / len(y))

Most common target words:
'.': 87
'’': 77
',': 43
'the': 34
'a': 30
'i': 28
'and': 22
'to': 22
'was': 17
't': 17
'you': 16
'of': 12
'me': 12
'it': 12
'he': 12

Comma target frequency: 0.0387038703870387


In [69]:
# Check the model's predictions and confidence
for context in ["the", "instead", "it was a"]:
    x = context_to_vector(context.split(), embedding_model).reshape(1, -1)
    probs, _ = forward_propagation(x, params)

    top_indices = np.argsort(probs[0])[-5:][::-1]

    print(f"\nContext: {context}")
    for idx in top_indices:
        print(f"{idx_to_word[int(idx)]!r}: {probs[0][idx]:.6f}")


Context: the
'matter': 0.003505
'chapter': 0.003386
',': 0.003257
'using': 0.003230
'“': 0.003228


KeyError: "Key 'instead' not present"

In [26]:
# Reinitialize the model with the current layer sizes
input_size = X.shape[1]
hidden1_size = 128
hidden2_size = 128
output_size = len(word_to_idx)

params = initialize_parameters(
    input_size, hidden1_size, hidden2_size, output_size
)

alpha = 0.01
costs = []

for epoch in range(1000):
    A3, cache = forward_propagation(X, params)
    cost = compute_cost(y, A3)
    costs.append(cost)

    grads = backpropagation(
        params["w1"], params["b1"],
        params["w2"], params["b2"],
        params["w3"], params["b3"],
        cache, X, y, alpha
    )

    dw1, db1, dw2, db2, dw3, db3 = grads

    params["w1"] -= alpha * dw1
    params["b1"] -= alpha * db1
    params["w2"] -= alpha * dw2
    params["b2"] -= alpha * db2
    params["w3"] -= alpha * dw3
    params["b3"] -= alpha * db3

    if epoch % 100 == 0:
        print(f"Epoch {epoch}: loss = {cost:.4f}")

print(f"Final loss: {costs[-1]:.4f}")

plt.plot(costs)
plt.xlabel("Epoch")
plt.ylabel("Cross-entropy loss")
plt.title("Training Loss")
plt.show()

Epoch 0: loss = 5.9179
Epoch 100: loss = 5.8826
Epoch 200: loss = 5.8474
Epoch 300: loss = 5.8085
Epoch 400: loss = 5.7629
Epoch 500: loss = 5.7066
Epoch 600: loss = 5.6353
Epoch 700: loss = 5.5474
Epoch 800: loss = 5.4536
Epoch 900: loss = 5.3801
Final loss: 5.3334


NameError: name 'plt' is not defined

In [222]:
for context in ["the", "he was", "it was a"]:
    print(context, "->", predict_next_word(context))

the -> .
he was -> .
it was a -> .


In [77]:
# Predictions on all training examples
A3, _ = forward_propagation(X, params)
predictions = np.argmax(A3, axis=1)

accuracy = np.mean(predictions == y)
print("Training accuracy:", accuracy)

# Most common predicted words
unique, counts = np.unique(predictions, return_counts=True)
top = np.argsort(counts)[::-1][:10]

print("\nMost predicted words:")
for i in top:
    idx = unique[i]
    print(idx_to_word[idx], counts[i], f"({counts[i] / len(y):.1%})")

# Most common actual target words
unique_y, counts_y = np.unique(y, return_counts=True)
top_y = np.argsort(counts_y)[::-1][:10]

print("\nMost common actual targets:")
for i in top_y:
    idx = unique_y[i]
    print(idx_to_word[idx], counts_y[i], f"({counts_y[i] / len(y):.1%})")

Training accuracy: 0.053058387395736796

Most predicted words:
, 4316 (100.0%)

Most common actual targets:
, 229 (5.3%)
the 169 (3.9%)
. 167 (3.9%)
of 99 (2.3%)
-- 97 (2.2%)
to 97 (2.2%)
i 90 (2.1%)
a 81 (1.9%)
and 75 (1.7%)
it 72 (1.7%)


In [54]:
import numpy as np

# Start with fresh parameters
params = initialize_parameters(
    X.shape[1], 128, 128, len(word_to_idx)
)

# Adam settings
learning_rate = 0.05
beta1 = 0.9
beta2 = 0.999
epsilon = 1e-8

keys = ["w1", "b1", "w2", "b2", "w3", "b3"]

m = {k: np.zeros_like(params[k]) for k in keys}
v = {k: np.zeros_like(params[k]) for k in keys}

costs = []
epochs = 600

for epoch in range(1, epochs + 1):
    A3, cache = forward_propagation(X, params)
    cost = compute_cost(y, A3)
    costs.append(cost)

    grads = backpropagation(
        params["w1"], params["b1"],
        params["w2"], params["b2"],
        params["w3"], params["b3"],
        cache, X, y, learning_rate
    )

    grads = dict(zip(keys, grads))

    for k in keys:
        m[k] = beta1 * m[k] + (1 - beta1) * grads[k]
        v[k] = beta2 * v[k] + (1 - beta2) * grads[k]**2

        m_hat = m[k] / (1 - beta1**epoch)
        v_hat = v[k] / (1 - beta2**epoch)

        params[k] -= learning_rate * m_hat / (
            np.sqrt(v_hat) + epsilon
        )

    if epoch == 1 or epoch % 100 == 0:
        print(f"Epoch {epoch}: loss = {cost:.4f}")

# Evaluate training accuracy
A3, _ = forward_propagation(X, params)
predictions = np.argmax(A3, axis=1)

print("Training accuracy:", np.mean(predictions == y))

unique, counts = np.unique(predictions, return_counts=True)
top = np.argsort(counts)[::-1][:10]

print("\nMost predicted words:")
for i in top:
    idx = unique[i]
    print(f"{idx_to_word[idx]}: {counts[i]} ({counts[i]/len(y):.1%})")

Epoch 1: loss = 6.3569
Epoch 100: loss = 4.2611
Epoch 200: loss = 3.8984
Epoch 300: loss = 3.7319
Epoch 400: loss = 3.6701
Epoch 500: loss = 3.4940
Epoch 600: loss = 3.4927
Training accuracy: 0.1539491298527443

Most predicted words:
.: 812 (36.2%)
’: 343 (15.3%)
t: 168 (7.5%)
to: 136 (6.1%)
i: 118 (5.3%)
was: 86 (3.8%)
s: 58 (2.6%)
mistakes: 30 (1.3%)
really: 19 (0.8%)
the: 18 (0.8%)


In [55]:
print("the ->", predict_next_word("the"))
print("he was ->", predict_next_word("he was"))
print("it was a ->", predict_next_word("it was a"))

the -> cars
he was -> ’
it was a -> take


In [58]:
def generate_text(start_text, num_words=50):
    words = start_text.lower().split()

    for _ in range(num_words):
        context = words[-3:]
        next_word = predict_next_word(" ".join(context))
        words.append(next_word)

    return " ".join(words)

print(generate_text("ed", 50))

ValueError: Unknown vocabulary words: ['baby']

In [43]:
embedding_model.wv["baby"]

KeyError: "Key 'baby' not present"

In [34]:
def show_top_predictions(context, top_n=5):
    words = context.lower().split()

    unknown = [w for w in words if w not in word_to_idx]
    if unknown:
        print("Unknown words:", unknown)
        return

    x = context_to_vector(words, embedding_model).reshape(1, -1)
    probabilities, _ = forward_propagation(x, params)

    top_indices = np.argsort(probabilities[0])[::-1][:top_n]

    print(f"\nContext: {context}")
    for idx in top_indices:
        print(
            f"{idx_to_word[idx]:<12} "
            f"{probabilities[0][idx]:.4f}"
        )

show_top_predictions("the")
show_top_predictions("he was")
show_top_predictions("it was a")
show_top_predictions("once upon a time")


Context: the
desultory    0.0724
mere         0.0719
sensation    0.0716
younger      0.0687
next         0.0679

Context: he was
was          0.1075
silent       0.0522
stood        0.0517
dead         0.0345
had          0.0314

Context: it was a
was          0.1100
his          0.0480
the          0.0414
not          0.0382
might        0.0224

Context: once upon a time
,            0.0928
more         0.0517
open         0.0448
.            0.0421
i            0.0410


In [35]:
WINDOW_SIZE = 3
EMBED_SIZE = embedding_model.wv.vector_size

def context_to_vector(context, embedding_model):
    context = context[-WINDOW_SIZE:]

    vectors = [
        embedding_model.wv[word]
        if word in embedding_model.wv
        else np.zeros(EMBED_SIZE)
        for word in context
    ]

    # Pad missing positions on the left
    while len(vectors) < WINDOW_SIZE:
        vectors.insert(0, np.zeros(EMBED_SIZE))

    return np.concatenate(vectors)